In [ ]:
# pyright: reportAttributeAccessIssue=false, reportUnusedFunction=false, reportUnusedImport=false, reportWildcardImportFromLibrary=false, reportUndefinedVariable=false, reportArgumentType=false
%load_ext autoreload
%autoreload 2

# core.domclass

>DOM Model with Pydantic and Pandoc Integration
output-file: core.domclass.html
title: core.domclass

This notebook demonstrates a Document Object Model (DOM) using Pydantic for static typing and validation, and integrates Pandoc (via pypandoc) for Markdown processing.
---

In [ ]:
# | default_exp core.domclass

In [ ]:
# | export

In [ ]:
#| export
from fastcore.basics import patch


In [ ]:
# | export
from collections.abc import Callable
from pathlib import Path
import hashlib
import json

import chromadb
import pypandoc
from chromadb import PersistentClient
from chromadb.api import ClientAPI
from chromadb.config import Settings


In [ ]:
#| export
from ollama import AsyncClient
from openai import AsyncOpenAI


In [ ]:
# | export
import copy
import os
import re
from typing import ClassVar, Optional, Any, TYPE_CHECKING
from pathlib import Path

import jsoncfg
from jsoncfg.config_classes import (
    ConfigJSONArray,
    ConfigJSONObject,
    ConfigJSONScalar,
)
from pydantic import BaseModel, Field
from ribosome.core.dom.summary import (
        AnalysisStatus, 
        summary_node_pair_async, 
        summary_node_async, 
        _summarize_leaf_async,
)
from ribosome.core.dom.embedding import embed
from ribosome.core.dom.utils import image_link_pattern, get_text_summary_response_async
from ribosome.core.dom.reorg import reorg_slides, reorg_node
from ribosome.core.dom.tree import map_tree

## DOM Class with pypandoc Integration

In [ ]:
# | export
class DOMClass(BaseModel):
    model_config = {"arbitrary_types_allowed": True}
    
    # The content of the Markdown document. This can be a string containing Markdown syntax.
    raw_markdown: Optional[str] = Field(None, description="Raw Markdown content")
    # raw json
    raw_json: Optional[str] = Field(None, description="Raw JSON content")
    # The json representation of the Markdown AST.
    ast_json: Optional[str] = Field(None, description="JSON representation of the Markdown AST")
    ast_json_file: Optional[Path] = Field(None, description="Path to the JSON file of the Markdown AST")
    semantics_json: Optional[str] = Field(None, description="Semantics JSON representation of the Markdown AST")
    semantics_json_file: Optional[Path] = Field(None, description="Path to the JSON file of the Markdown AST semantics")
    embed_json: Optional[str] = Field(None, description="Semantics JSON representation with embeddings and meta information of the Markdown AST")
    embed_json_file: Optional[Path] = Field(None, description="Path to the JSON file of the Markdown AST embeddings")
    file_path: Optional[Path] = Field(None, description="Path to the Markdown file")
    root_path: Path = Field(default_factory=Path, description="root path of the markdown document, required to get access to the images")
    table_count: int = Field(0, description="Number of tables in the Markdown document")
    section_count: int = Field(0, description="Number of headers in the Markdown document")
    section_level: list = Field(default_factory=list, description="List of section levels in the Markdown document")
    title:str = Field('', description="Title of the Markdown document, if available")
    ollama_client: AsyncClient = Field(default_factory=AsyncClient, description="AsyncClient instance for embeddings and image analysis (Ollama)")
    ollama_model: str = Field("gemme4:e4b", description="Model name for the Ollama client")
    openai_client: AsyncOpenAI = Field(default_factory=lambda: AsyncOpenAI(api_key=os.environ.get('DASHSCOPE_API_KEY'), base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"), description="AsyncOpenAI instance for text summarization (OpenAI-compatible APIs)")
    openai_model: str = Field("qwen3.6-flash")
    db_client: ClientAPI = Field(default_factory=PersistentClient, description="PersistentClient instance for database interactions")
    analysis_status: AnalysisStatus = Field(
        default_factory=lambda: AnalysisStatus(status=ResponseStatus.PENDING, exception=""),
        description="Analysis status of the Markdown document",
    )


    # Use ClassVar to indicate these are class variables, not instance fields
    TextBlock_Types: ClassVar[set[str]] = {
        "Plain",
        "Para",
        "Figure",
        "LineBlock","CodeBlock","RawBlock","OrderedList","BulletList","DefinitionList",
        "Header","BlockQuote",
        "Table","TableRow", "TableCell"}

    NonTextBlock_Types: ClassVar[set[str]] = {"HorizontalRule", "Div", "Null"}

    Embed_Types: ClassVar[set[str]] = {"Section", "Image", "Table"}

    Block_Types: ClassVar[set[str]] = TextBlock_Types.union(TextBlock_Types)

    Inline_Types: ClassVar[set[str]] = {
        "Str", "Emph", "Strong", "Strikeout", "Superscript", "Subscript",
        "Decimal", "Period",
        "Link", 
        "Image", "Code", "Math", "RawInline", "SoftBreak", "HardBreak", "Span"   
    }
    
    Element_Types: ClassVar[set[str]] = Block_Types | Inline_Types
    leaf_min_len: ClassVar[int] = 100  # Minimum length of text to consider for summarization
    min_len: ClassVar[int] = 300  # Minimum length of the final summary
    lang: ClassVar[str] = "zh"  # Language of the document

    _ARTIFACT_SUFFIXES: ClassVar[dict[str, str]] = {
        "ast_json_file": "_ast.json",
        "semantics_json_file": "_semantics.json",
        "embed_json_file": "_embed.json",
    }

    if TYPE_CHECKING:
    #     async def _map_tree_async(self, _node: Any, _transform: Callable, _on_exit: Optional[Callable] = None) -> Any: ...
    #     def _config_node_line(self, _config_node: ConfigNode) -> int | str: ...
    #     def _config_error(self, _message: str, _config_node: ConfigNode, _node: Any = None) -> ValueError: ...
    #     async def _summarize_leaf_async(self, _node: str, _min_len: Optional[int] = None) -> str: ...
    #     async def _coerce_summary_async(self, _value: Any) -> str: ...
    #     async def _summarize_list_async(self, _root: list) -> str: ...
        def _document_metadata(self) -> dict[str, str]: ...
        async def _finalize_summary_ast_async(self, _ast_dict: dict, _blocks: list) -> str: ...
    #     def _record_summary_progress(self, _node_type: str, _config_node: ConfigNode) -> None: ...
    #     def _resolve_image_link(self, _node: dict, _config_node: ConfigNode) -> str: ...
    #     async def _summarize_image_node_async(self, _node: dict, _config_node: ConfigNode) -> dict: ...
        def reorg(self, _action: Optional[Callable] = None) -> str: ...
    #     def reorg_section(self, _section: dict, _it: Iterator, _bIgnoreLevel: bool = False) -> tuple[dict, Iterator]: ...
        @classmethod
        def identity(cls, _obj: Any) -> Any: ...
    

    def __init__(self, md_file_path: Path, ollama_client: Optional[AsyncClient] = None, openai_client: Optional[AsyncOpenAI] = None, db_client: Optional[ClientAPI] = None, **data):
        """Initialize a DOMClass document wrapper for a markdown file.
        
        Args:
            md_file_path: Path to the source markdown file.
            ollama_client: Optional async Ollama client override.
            openai_client: Optional async OpenAI-compatible client override.
            db_client: Optional ChromaDB client override.
            **data: Additional model fields passed to `BaseModel`.
        """
        # Set defaults for data if not provided
        data.setdefault('file_path', Path(md_file_path))
        data.setdefault('root_path', Path(md_file_path).parent)
        data.setdefault('title', Path(md_file_path).stem)
        data.setdefault('ollama_client', ollama_client or AsyncClient())
        data.setdefault('ollama_model', "gemme4:e4b")
        data.setdefault('openai_client', openai_client or AsyncOpenAI(api_key=os.environ.get('DASHSCOPE_API_KEY'), base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"))
        data.setdefault('openai_model', "qwen3.6-flash")
        data.setdefault('db_client', db_client or PersistentClient())
        data.setdefault('ast_json_file', None)
        data.setdefault('embed_json_file', None)
        data.setdefault('semantics_json_file', None)

        super().__init__(**data)

    def _artifact_path(self, suffix: str) -> Path:
        assert self.file_path is not None, "file_path is required to build artifact paths"
        return self.file_path.parent / f"{self.file_path.stem}{suffix}"

    def _sync_artifact_fields(self) -> None:
        for attr_name, suffix in self._ARTIFACT_SUFFIXES.items():
            setattr(self, attr_name, self._artifact_path(suffix))

    def _load_artifact(self, path_attr: str, content_attr: str) -> None:
        path = getattr(self, path_attr)
        if path and path.exists():
            setattr(self, content_attr, path.read_text(encoding="utf-8"))

    def _reset_document_state(self) -> None:
        self.raw_markdown = None
        self.raw_json = None
        self.ast_json = None
        self.semantics_json = None
        self.embed_json = None
        self.ast_json_file = None
        self.semantics_json_file = None
        self.embed_json_file = None

    def _require_action(self, action: Optional[Callable]) -> Callable:
        return action or self.__class__.identity

    def _render_markdown(self, target: str) -> str | None:
        if not self.raw_markdown:
            return None
        return pypandoc.convert_text(self.raw_markdown, target, "md")

    def _dump_json(self, value: dict | list) -> str:
        return json.dumps(value, ensure_ascii=False).encode("utf-8").decode("utf-8")


In [ ]:
# | export
@patch
def _document_metadata(self: DOMClass) -> dict[str, str]:
    """Get the document metadata.

    Args:
        self (DOMClass): The DOMClass instance.

    Returns:
        dict[str, str]: The document metadata.
    """
    if self.title is None:
        self.title = str(self.root_path)
    return {
        'title': self.title,
        'file_path': str(self.file_path),
    }


In [ ]:
# | export
@patch
async def _finalize_summary_ast_async(self: DOMClass, ast_dict: dict, blocks: list) -> str:
    """Finalize the summary of the AST asynchronously.

    Args:
        self (DOMClass): The DOMClass instance.
        ast_dict (dict): The AST dictionary.
        blocks (list): The list of blocks to be summarized.

    Returns:
        str: The serialized summary of the AST.
    """
    ast_dict.update(self._document_metadata())
    dict_summary = [b['s'] for b in blocks if isinstance(b, dict) and 's' in b]
    dict_summary = [ast_dict['title']] + dict_summary
    ast_dict['summary'] = "" if not ast_dict['blocks'] or dict_summary == [] else await _summarize_leaf_async(
        " ".join(dict_summary), 
        leaf_min_len=self.leaf_min_len,
        client=self.ollama_client,
        model=self.ollama_model, 
        min_len=self.min_len,
        lang="zh")
    serialized = self._dump_json(ast_dict)
    self.ast_json = serialized
    return serialized


In [ ]:
# | export
@patch
def setup(self: DOMClass):
    """Load markdown content and initialize cached AST, semantics, and embedding artifacts.
    
    Args:
        None.
    """
    content = self.file_path.read_text(encoding="utf-8")  # type: ignore
    if not content:
        self._reset_document_state()
        return

    self.raw_markdown = content
    self.raw_json = pypandoc.convert_text(self.raw_markdown, "json", "md")
    self._sync_artifact_fields()

    if self.ast_json_file and self.ast_json_file.exists():
        self.ast_json = self.ast_json_file.read_text(encoding="utf-8")
    else:
        print(f"--- IGNORE --- {self.file_path}")
        slide_splitter = r"(^<!--\s*Slide number:\s*\d+\s*-->$)"
        has_slides = re.search(slide_splitter, self.raw_markdown, flags=(re.MULTILINE | re.IGNORECASE))
        self.ast_json = reorg_slides(raw_json=self.raw_markdown, 
                                    raw_markdown=self.raw_markdown,
                                    slide_splitter=slide_splitter) if has_slides else self.reorg()
        if self.ast_json_file and self.ast_json is not None:
            self.ast_json_file.write_text(self.ast_json, encoding="utf-8")

    self._load_artifact("semantics_json_file", "semantics_json")
    self._load_artifact("embed_json_file", "embed_json")


In [ ]:
# | export
@patch
def to_markdown(self: DOMClass) -> str | None:
    """Return the raw markdown content for the current document.
    
    Args:
        None.
    
    Returns:
        str | None: Stored markdown text, if available.
    """
    return self.raw_markdown


In [ ]:
# | export
@patch
def to_html(self: DOMClass) -> str | None:
    """Render the current markdown content to HTML.
    
    Args:
        None.
    
    Returns:
        str | None: Rendered HTML, or `None` when no markdown is loaded.
    """
    return self._render_markdown("html")


In [ ]:
# | export
@patch
def to_latex(self: DOMClass) -> str | None:
    """Render the current markdown content to LaTeX.
    
    Args:
        None.
    
    Returns:
        str | None: Rendered LaTeX, or `None` when no markdown is loaded.
    """
    return self._render_markdown("latex")


In [ ]:
# | export
@patch
def to_json(self: DOMClass) -> str | None:
    """Return the serialized Pandoc AST for the current document.
    
    Args:
        None.
    
    Returns:
        str | None: Stored AST JSON, if available.
    """
    return self.raw_json


In [ ]:
# | export
@patch
async def embed_doc(self: DOMClass, action: Optional[Callable] = None, db_path: Optional[Path] = Path("../db")) -> None:
    """Embed summarized semantic nodes and persist the embedding JSON output.
    
    Args:
        action: Optional transform applied to each node before traversal.
        db_path: Path to the persistent Chroma database directory.
    
    Returns:
        None
    """
    if not self.semantics_json:
        raise ValueError("semantics_json content is empty. Cannot embed the content.")
    action = self._require_action(action)

    if self.embed_json:
        # If embed_json is already set, we don't need to walk the AST again
        print("embed_json is already set. Skipping walk.")
        return

    try:
        self.db_client = PersistentClient(path=str(db_path), settings=Settings(allow_reset=True))  # type: ignore
        # ephemeral_db_client = chromadb.EphemeralClient()  # type: ignore     
        # assert ephemeral_db_client, "Failed to create ephemeral ChromaDB client."
    except Exception as e:
        print(f"Failed to create PersistentClient: {e}")
    # Create or get the collection in the database 'mitochondria'
    collection = self.db_client.get_or_create_collection(name="mitochondria")  # defined in the closure for embed_node
    # collection = ephemeral_db_client.get_or_create_collection(name="mitochondria")
    assert collection, "Failed to create or get the 'mitochondria' collection in the database."        


    assert self.file_path, "file path is not set."
    self.embed_json = await embed(
        semantics_json=self.semantics_json,
        file_path=self.file_path,
        embed_types=self.Embed_Types,
        db_client=self.db_client,
        collection=collection,
        llm_client=self.ollama_client,
        model=self.ollama_model,
        action=action,
    )

    embed_json_file = self.file_path.parent / f"{self.file_path.stem}_embed.json"
    assert isinstance(self.embed_json, str), "embed_json should be a string."
    with open(embed_json_file, "w", encoding="utf-8") as f:
        f.write(self.embed_json)

In [ ]:
# | export
@patch
def reorg(self: DOMClass, action: Optional[Callable] = None) -> str:
    """Convert header-based AST blocks into nested section structures.
    
    Args:
        action: Optional transform applied to each visited node.
    
    Returns:
        str: Reorganized AST serialized as JSON.
    """
    if not self.raw_json:
        raise ValueError("raw_json content is empty. Cannot walk the AST.")
    action = self._require_action(action)

    ast = json.loads(self.raw_json)

    ast = reorg_node(
        node=ast,
        section_level=self.section_level,
        action=self.identity
    )
    return self._dump_json(ast)

In [ ]:
# | export
@patch
async def textualize(self: DOMClass, action: Optional[Callable] = None) -> None:
    """Summarize AST nodes into semantic text fields for downstream embedding.
    
    Args:
        action: Optional transform applied to each visited node.
    """
    if not self.ast_json:
        raise ValueError("raw_json content is empty. Cannot walk the AST and summarize.")
    action = self._require_action(action)


    config_ast = jsoncfg.load_config(str(self.ast_json_file))
    ast_dict = json.loads(self.ast_json)
    if not isinstance(config_ast, ConfigJSONObject):
        raise ValueError(f"AST should be a dictionary, got {type(config_ast)}")

    config_blocks = config_ast['blocks']
    blocks = ast_dict['blocks']
    assert isinstance(self.file_path, Path), "self.file_path should be a Path object"
    _, blocks = await summary_node_pair_async(
        config_node=config_blocks, 
        node=blocks, 
        root_path=
        self.root_path,client=self.ollama_client,
        model="gemma4:e4b",
        file_path=self.file_path,
        table_count=self.table_count,
        section_count=self.section_count
        )
    ast_dict['blocks'] = blocks
    assert isinstance(blocks,list)
    await self._finalize_summary_ast_async(ast_dict, blocks)
    return

    # # Run the summary_nodewline_main function asynchronously
    # # asyncio.run(summary_nodewline_main(action))
    # await summary_nodewline_main(action)
    # return 


In [ ]:
# | export
@patch
def identity(self: DOMClass, obj: Any) -> Any:
    """Return an object unchanged during traversal hooks.
    
    Args:
        obj: Object to return as-is.
    
    Returns:
        Any: The original object.
    """
    return obj


In [ ]:
# | export
@patch
async def summary_nodewline_main(self: DOMClass,action: Optional[Callable] = None) -> None:
    """
    Main function to walk the AST and summarize nodes.
    This function will be called by the walk method.
    """

    if not self.ast_json:
        raise ValueError("raw_json content is empty. Cannot walk the AST and summarize.")
    if action is None:
        action = self.__class__.identity
    config_ast = jsoncfg.load_config(str(self.ast_json_file))
    ast_dict = json.loads(self.ast_json)
    assert isinstance(config_ast, ConfigJSONObject), f"AST should be a dictionary, got {type(config_ast)}"
    config_blocks = config_ast['blocks']
    blocks = ast_dict['blocks']
    assert self.ast_json_file, "ast_json_file not ready!"
    _ , blocks = await summary_node_pair_async(
        config_node=config_blocks, 
        node=blocks, 
        root_path=self.root_path,
        file_path=self.ast_json_file,
        client=self.ollama_client,
        model=self.ollama_model,
        table_count=self.table_count,
        section_count=self.section_count,
        action=self.identity,
        leaf_min_len=self.leaf_min_len,
        min_len=self.min_len,
        lang=self.lang)
    ast_dict['blocks'] = blocks
    if ast_dict.get('title') is None:
        # If the title is not set, use the file name as the title
        if self.title is None:
            self.title = str(self.root_path)
    ast_dict['title'] = self.title  # Add the title to the AST
    ast_dict['file_path'] = str(self.file_path)  # Add the file path to the AST
    assert isinstance(blocks, list), f"Expected a list of blocks, got {type(blocks)}"
    dict_summary = [b['s'] for b in blocks if isinstance(b, dict) and 's' in b]
    dict_summary = [ast_dict['title']] + dict_summary  # Add the title to the summary list
    # the summary of the document from the summaries in the list of blocks
    if not ast_dict['blocks'] or dict_summary == []:
        # If the blocks are empty, set the summary to an empty string
        ast_dict['summary'] = ""
        # If the summary is empty, set the AST JSON to an empty string
        self.ast_json = json.dumps(ast_dict, ensure_ascii=False).encode("utf-8").decode("utf-8")
    else:
        # If the blocks are not empty, set the summary to the concatenated summaries
        doc_summary = await _summarize_leaf_async(
            node=" ".join(dict_summary),
            leaf_min_len=self.leaf_min_len,
            min_len=self.min_len,
            client=self.ollama_client,
            model=self.ollama_model,
            lang=self.lang
            )
        ast_dict['summary'] = doc_summary
        # Convert the summarized AST back to JSON
        self.ast_json = json.dumps(ast_dict, ensure_ascii=False).encode("utf-8").decode("utf-8")

In [ ]:
# | export
@patch
async def summary_node_main(self: DOMClass, action: Optional[Callable] = None) -> None:
    """
    Main function to walk the AST and summarize nodes.
    This function will be called by the walk method.
    """

    if not self.ast_json:
        raise ValueError("raw_json content is empty. Cannot walk the AST and summarize.")
    if action is None:
        action = self.__class__.identity
    ast = json.loads(self.ast_json)
    blocks = ast.get("blocks", [])
    assert isinstance(self.file_path, Path), "self.file_path should be a Path object"
    ast['blocks'] = await summary_node_async(
        node=blocks,
        root_path=self.root_path,
        file_path=self.file_path,
        client=self.ollama_client,
        model=self.ollama_model,
        table_count=self.table_count,
        section_count=self.section_count,
        action=self.identity,
        leaf_min_len=self.leaf_min_len,
        min_len=self.min_len,
        lang=self.lang
        )
    if ast.get('title') is None:
        # If the title is not set, use the file name as the title
        if self.title is None:
            self.title = str(self.root_path)
    ast['title'] = self.title  # Add the title to the AST
    ast['file_path'] = str(self.file_path)  # Add the file path to the AST
    assert isinstance(blocks, list), f"Expected a list of blocks, got {type(blocks)}"
    dict_summary = [b['s'] for b in blocks if isinstance(b, dict) and 's' in b]
    dict_summary = [ast['title']] + dict_summary  # Add the title to the summary list
    # the summary of the document from the summaries in the list of blocks
    if not ast['blocks'] or dict_summary == []:
        # If the blocks are empty, set the summary to an empty string
        ast['summary'] = ""
        # If the summary is empty, set the AST JSON to an empty string
        self.ast_json = json.dumps(ast, ensure_ascii=False).encode("utf-8").decode("utf-8")
    else:
        # If the blocks are not empty, set the summary to the concatenated summaries
        doc_summary = await _summarize_leaf_async(
            node=" ".join(dict_summary),
            leaf_min_len=self.leaf_min_len,
            client=self.ollama_client,
            model=self.ollama_model,
            min_len=self.min_len,
            lang=self.lang
        )
        ast['summary'] = doc_summary
        # Convert the summarized AST back to JSON
        self.ast_json = json.dumps(ast, ensure_ascii=False).encode("utf-8").decode("utf-8")

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

## DOM Tests

Fastcore-style checks for the DOM helpers, traversal methods, setup/reorg flow, textualization, and embedding behavior.

In [ ]:
# | hide
import importlib.util
import io
import sys
import tempfile
from contextlib import redirect_stdout
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

from fastcore.test import test_eq

repo_root = Path.cwd()
if not (repo_root / "ribosome/core/domclass.py").exists():
    repo_root = repo_root.parent
assert (repo_root / "ribosome/core/domclass.py").exists(), "Unable to locate ribosome/core/domclass.py"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

dom_module_path = repo_root / "ribosome/core/domclass.py"
dom_spec = importlib.util.spec_from_file_location("ribosome.core._dom_impl", dom_module_path)
assert dom_spec is not None and dom_spec.loader is not None
dom_module = importlib.util.module_from_spec(dom_spec)
dom_spec.loader.exec_module(dom_module)
DOMClass = dom_module.DOMClass
ResponseStatus = dom_module.ResponseStatus


class FakeEmbedResponse:
    def __init__(self, text: str):
        self.embedding = [float(len(text or ""))]
        self.model = "fake-embed-model"


class FakeOllamaClient:
    async def embed(self, model: str, input: str):
        return FakeEmbedResponse(input)


class FakeCollection:
    def __init__(self):
        self.records = []

    def add(self, ids, metadatas, embeddings, documents):
        self.records.append(
            {
                "ids": ids,
                "metadatas": metadatas,
                "embeddings": embeddings,
                "documents": documents,
            }
        )


class FakeDbClient:
    def __init__(self, collection: FakeCollection):
        self.collection = collection

    def get_or_create_collection(self, name: str):
        return self.collection


def make_dom_fixture(content: str = "# Title\n\nParagraph\n") -> tuple[tempfile.TemporaryDirectory, Path, Path, DOMClass]:
    tmpdir = tempfile.TemporaryDirectory()
    root = Path(tmpdir.name)
    md_file = root / "doc.md"
    md_file.write_text(content, encoding="utf-8")
    dom = DOMClass(md_file)
    object.__setattr__(dom, "ollama_client", FakeOllamaClient())
    object.__setattr__(dom, "openai_client", SimpleNamespace(name="fake-openai"))
    object.__setattr__(dom, "db_client", SimpleNamespace(name="fake-db"))
    return tmpdir, root, md_file, dom

In [ ]:
# | hide
tmpdir, root, md_file, dom = make_dom_fixture()
print(f"tmpdir: {tmpdir}; root: {root}, md_file: {md_file}, dom: {dom}")
test_eq(dom.file_path, md_file)
test_eq(dom.root_path, root)
test_eq(dom.title, "doc")
test_eq(dom.analysis_status.status, ResponseStatus.PENDING)
test_eq(dom.analysis_status.exception, "")
test_eq(DOMClass.identity("token"), "token")

artifact_path = dom._artifact_path("_ast.json")
test_eq(artifact_path, root / "doc_ast.json")
dom._sync_artifact_fields()
test_eq(dom.ast_json_file, root / "doc_ast.json")
test_eq(dom.semantics_json_file, root / "doc_semantics.json")
test_eq(dom.embed_json_file, root / "doc_embed.json")

dom.ast_json_file.write_text('{"blocks": [1]}', encoding="utf-8")
dom._load_artifact("ast_json_file", "ast_json")
test_eq(dom.ast_json, '{"blocks": [1]}')

def marker(node):
    if isinstance(node, dict) and node.get("t") == "Str":
        updated = dict(node)
        updated["visited"] = True
        return updated
    return node

mapped = dom._map_tree({"items": [{"t": "Str", "c": "hello"}]}, marker)
test_eq(mapped["items"][0]["visited"], True)

with patch.object(pypandoc, "convert_text", return_value="<p>Paragraph</p>"):
    dom.raw_markdown = "Paragraph"
    test_eq(dom._render_markdown("html"), "<p>Paragraph</p>")
    test_eq(dom.to_html(), "<p>Paragraph</p>")

with patch.object(pypandoc, "convert_text", return_value="\\paragraph{Paragraph}"):
    test_eq(dom.to_latex(), "\\paragraph{Paragraph}")

test_eq(dom.to_markdown(), "Paragraph")
dom.raw_json = '{"blocks": []}'
test_eq(dom.to_json(), '{"blocks": []}')
test_eq(dom._dump_json({"text": "你好"}), '{"text": "你好"}')

def identity_action(value):
    return value

test_eq(dom._require_action(identity_action), identity_action)
test_eq(dom._require_action(None), DOMClass.identity)

config_file = root / "config.json"
config_file.write_text('{"node": {"kind": "value"}}', encoding="utf-8")
config_ast = jsoncfg.load_config(str(config_file))
assert config_ast, "config_ast is None"
line_number = dom._config_node_line(config_ast["node"]["kind"])
test_eq(isinstance(line_number, int), True)
config_error = dom._config_error("boom", config_ast["node"]["kind"], {"kind": "value"})
test_eq("boom" in str(config_error), True)
test_eq("Node: {'kind': 'value'}" in str(config_error), True)

stdout = io.StringIO()
with redirect_stdout(stdout):
    dom._record_summary_progress("Table", config_ast["node"]["kind"])
    dom._record_summary_progress("Section", config_ast["node"]["kind"])
test_eq("Summarize table" in stdout.getvalue(), True)
test_eq("Summarize section" in stdout.getvalue(), True)
test_eq(dom.table_count, 1)
test_eq(dom.section_count, 1)

(root / "robot.png").write_bytes(b"png")
image_node = {"c": [["", [], []], [], ["robot.png", ""]]}
test_eq(dom._resolve_image_link(image_node, config_ast["node"]), str(root / "robot.png"))

dom.raw_markdown = "filled"
dom.raw_json = "filled"
dom.ast_json = "filled"
dom.semantics_json = "filled"
dom.embed_json = "filled"
dom._reset_document_state()
test_eq(dom.raw_markdown, None)
test_eq(dom.raw_json, None)
test_eq(dom.ast_json, None)
test_eq(dom.semantics_json, None)
test_eq(dom.embed_json, None)

tmpdir.cleanup()

tmpdir, root, md_file, dom = make_dom_fixture()
slide_section = {
    "t": "Section",
    "c": [
        {"t": "Header", "c": [1, ["s1", [], []], [{"t": "Str", "c": "Title"}]]},
        {"t": "Content", "c": []},
    ],
}
following = iter([
    {"t": "Para", "c": [{"t": "Str", "c": "Body"}]},
    {"t": "Section", "c": [
        {"t": "Header", "c": [1, ["s2", [], []], [{"t": "Str", "c": "Next"}]]},
        {"t": "Content", "c": []},
    ]},
])
packed, remaining = dom.reorg_section(copy.deepcopy(slide_section), following, bIgnoreLevel=False)
test_eq(len(packed["c"][1]["c"]), 1)
test_eq(list(remaining)[0]["t"], "Section")

dom.raw_json = json.dumps({
    "blocks": [
        {"t": "Header", "c": [1, ["intro", [], []], [{"t": "Str", "c": "Intro"}]]},
        {"t": "Para", "c": [{"t": "Str", "c": "Paragraph"}]},
    ]
}, ensure_ascii=False)
reorg_ast = json.loads(dom.reorg())
test_eq(reorg_ast["blocks"][0]["t"], "Section")
test_eq(reorg_ast["blocks"][0]["c"][1]["c"][0]["t"], "Para")

dom.raw_markdown = "<!-- Slide number: 1 -->\nSlide body"
dom.raw_json = json.dumps({"blocks": []}, ensure_ascii=False)
slide_ast_json = json.dumps({
    "blocks": [
        {"t": "Para", "c": [{"t": "Str", "c": "Slide body"}]},
    ]
}, ensure_ascii=False)
with patch.object(pypandoc, "convert_text", return_value=slide_ast_json):
    slide_ast = json.loads(dom.reorg_slides())
test_eq(slide_ast["blocks"][0]["t"], "Section")

dom.raw_json = json.dumps({"blocks": [{"t": "Str", "c": "hello"}]}, ensure_ascii=False)
dom.walk(marker)
walked = json.loads(dom.ast_json)
test_eq(walked["blocks"][0]["visited"], True)

dom.ast_json_file = root / "doc_ast.json"
dom.ast_json_file.write_text(json.dumps({"blocks": [{"t": "Para", "c": []}]}, ensure_ascii=False), encoding="utf-8")
dom.raw_json = dom.ast_json_file.read_text(encoding="utf-8")
walk_stdout = io.StringIO()
with redirect_stdout(walk_stdout):
    dom.walk_nodes_with_line_number()
test_eq("key" in walk_stdout.getvalue(), True)

tmpdir.cleanup()

tmpdir, root, md_file, dom = make_dom_fixture("# Title\n")
artifact_ast = root / "doc_ast.json"
artifact_semantics = root / "doc_semantics.json"
artifact_embed = root / "doc_embed.json"
artifact_ast.write_text('{"blocks": ["cached"]}', encoding="utf-8")
artifact_semantics.write_text('{"summary": "semantic"}', encoding="utf-8")
artifact_embed.write_text('{"summary": "embed"}', encoding="utf-8")
with patch.object(pypandoc, "convert_text", return_value='{"blocks": ["raw"]}'):
    dom.setup()
test_eq(dom.raw_markdown, "# Title\n")
test_eq(dom.raw_json, '{"blocks": ["raw"]}')
test_eq(dom.ast_json, '{"blocks": ["cached"]}')
test_eq(dom.semantics_json, '{"summary": "semantic"}')
test_eq(dom.embed_json, '{"summary": "embed"}')
test_eq(dom.ast_json_file, artifact_ast)

tmpdir.cleanup()

tmpdir, root, md_file, dom = make_dom_fixture("")
dom.raw_markdown = "set"
dom.raw_json = "set"
dom.setup()
test_eq(dom.raw_markdown, None)
test_eq(dom.raw_json, None)
tmpdir.cleanup()

In [ ]:
# | hide
tmpdir, root, md_file, dom = make_dom_fixture()

visited = []
exited = []

async def mark_async(node):
    if isinstance(node, dict) and node.get("t") == "Str":
        updated = dict(node)
        updated["async_seen"] = True
        visited.append(updated["c"])
        return updated
    return node

async def on_exit(node):
    if isinstance(node, dict) and node.get("t") == "Str":
        exited.append(node["c"])

async_tree = await dom._map_tree_async({"items": [{"t": "Str", "c": "hello"}]}, mark_async, on_exit=on_exit)
test_eq(async_tree["items"][0]["async_seen"], True)
test_eq(visited, ["hello"])
test_eq(exited, ["hello"])

async def fake_summary(client, text, model="unused", role="user", lang="zh"):
    return f"SUM<{text}>"

with patch.dict(DOMClass._summarize_leaf_async.__globals__, {"get_summary_response_async": fake_summary}):
    test_eq(await dom._summarize_leaf_async("short", min_len=10), "short")
    test_eq(await dom._summarize_leaf_async("long text", min_len=3), "SUM<long text>")

async def fake_leaf(self, node, min_len=None):
    return f"S<{node}>"

with patch.object(DOMClass, "_summarize_leaf_async", new=fake_leaf):
    test_eq(await dom._coerce_summary_async("x"), "S<x>")
    test_eq(await dom._coerce_summary_async(7), "S<7>")
    test_eq(await dom._coerce_summary_async({"s": "done"}), "done")
    test_eq(await dom._coerce_summary_async(None), "")
    test_eq(await dom._summarize_list_async(["a", 2, {"s": "b"}]), "S<S<a> S<2> b>")

coerce_error = None
try:
    await dom._coerce_summary_async(object())
except ValueError as exc:
    coerce_error = exc
test_eq("Unsupported element type" in str(coerce_error), True)

metadata = dom._document_metadata()
test_eq(metadata["title"], "doc")
test_eq(metadata["file_path"], str(md_file))

with patch.object(DOMClass, "_summarize_leaf_async", new=fake_leaf):
    finalized = await dom._finalize_summary_ast_async({"blocks": [{"s": "alpha"}]}, [{"s": "alpha"}])
finalized_dict = json.loads(finalized)
test_eq(finalized_dict["title"], "doc")
test_eq(finalized_dict["summary"], "S<doc alpha>")
test_eq(dom.ast_json, finalized)

config_file = root / "image_config.json"
config_file.write_text('{"image": {"c": [["", [], []], [], ["robot.png", ""]]}}', encoding="utf-8")
config_ast = jsoncfg.load_config(str(config_file))
image_node = {"c": [["", [], []], [], ["robot.png", ""]]}
(root / "robot.png").write_bytes(b"img")

async def fake_image_summary(client, image_link, model="unused", role="user", lang="zh"):
    return f"IMG<{Path(image_link).name}>"

with patch.object(dom_module, "get_image_summary_async", new=fake_image_summary):
    image_result = await dom._summarize_image_node_async(image_node, config_ast["image"])
test_eq(image_result["s"], "IMG<robot.png>")

tmpdir.cleanup()

tmpdir, root, md_file, dom = make_dom_fixture()
dom.ast_json_file = root / "doc_ast.json"
ast_dict = {
    "blocks": [
        {
            "t": "Section",
            "c": [
                {"t": "Header", "c": [1, ["intro", [], []], [{"t": "Str", "c": "Intro"}]]},
                {"t": "Content", "c": []},
            ],
        }
    ]
}
dom.ast_json = json.dumps(ast_dict, ensure_ascii=False)
dom.ast_json_file.write_text(dom.ast_json, encoding="utf-8")

async def fake_summary_pair(config_blocks, blocks):
    blocks[0]["s"] = "section summary"
    return config_blocks, blocks

with patch.object(dom_module, "summary_node_pair_async", new=fake_summary_pair), patch.object(DOMClass, "_summarize_leaf_async", new=fake_leaf):
    await dom.textualize()
textualized = json.loads(dom.ast_json)
test_eq(textualized["blocks"][0]["s"], "section summary")
test_eq(textualized["summary"], "S<doc section summary>")

dom.semantics_json = json.dumps(
    {
        "summary": "document summary",
        "blocks": [
            {
                "t": "Section",
                "s": "section one",
                "c": [
                    {"t": "Header", "c": [1, ["s1", [], []], [{"t": "Str", "c": "S1"}]]},
                    {"t": "Content", "c": [{"t": "Table", "s": "table one", "c": []}]},
                ],
            },
            {
                "t": "Section",
                "s": "section two",
                "c": [
                    {"t": "Header", "c": [1, ["s2", [], []], [{"t": "Str", "c": "S2"}]]},
                    {"t": "Content", "c": []},
                ],
            },
        ],
    },
    ensure_ascii=False,
 )

collection = FakeCollection()
with patch.dict(DOMClass.embed.__globals__, {"PersistentClient": lambda *args, **kwargs: SimpleNamespace()}), patch.object(chromadb, "EphemeralClient", return_value=FakeDbClient(collection)):
    embed_json = await dom.embed(db_path=root / "db")
embedded = json.loads(embed_json)
test_eq("i" in embedded, True)
test_eq(len(collection.records), 4)
root_path = collection.records[0]["metadatas"][0]["obj_path"]
section_one_path = collection.records[1]["metadatas"][0]["obj_path"]
table_path = collection.records[2]["metadatas"][0]["obj_path"]
section_two_path = collection.records[3]["metadatas"][0]["obj_path"]
test_eq(len(root_path), 1)
test_eq(section_one_path, root_path)
test_eq(len(table_path), 2)
test_eq(table_path[0], root_path[0])
test_eq(section_two_path, root_path)

tmpdir.cleanup()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()